In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from transformers import GPT2Tokenizer, GPT2Model
import pandas as pd
import numpy as np
import os
import pickle
import gc
import sys
import argparse
from torch.utils.data import DataLoader
from nltk.translate.bleu_score import sentence_bleu
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import torch
import skimage.io as io
import clip
from PIL import Image
import pickle
import json
import os
from tqdm import tqdm
import argparse
from sklearn.model_selection import train_test_split


tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2 = GPT2Model.from_pretrained('gpt2')

In [2]:
gpt2

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [3]:
import pandas as pd
dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
image_id_counts = data['image_id'].value_counts()

data.describe()
import pandas as pd

dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
image_id_counts = data['image_id'].value_counts()

data.describe()

,funny_score
count,3.265261e+06
mean,5.851492e-05
std,1.709560e-03
min,0.000000e+00
25%,0.000000e+00
50%,2.037739e-05
75%,5.094347e-05
max,1.000000e+00


In [239]:
# get 75% threshold of funny_score
threshold = data['funny_score'].quantile(0.75)
print(f'threshold: {threshold:.60f}')

threshold: 0.000050943473122223578502480029195353949944546911865472793579


# MiniGPT4

In [4]:
from string import punctuation

import pandas as pd
from scipy.special import huber
from sklearn.model_selection import train_test_split
import re


# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
dirPath = '../Data/Oxford_HIC/Minigpt4_Oxford.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# dirPath = '../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv'


data = pd.read_csv(dirPath)
data

,image_id,emotion,sentiment,humor
0,bokete_94229,seriousness;introspection,melancholy;reflection,amusement;irony
1,imgflip_0,joyful,energetic,playful
2,imgflip_1,sad;confused;happy,positive;negative;neutral,none;irony;sarcasm
3,imgflip_2,surprise;frustration;sadness;curiosity,playfulness;anticipation;melancholic,exaggeration;none;irony
4,imgflip_3,intensity;anger;violence;fear,heroism;villainy;gritty;dark,irony
...,...,...,...,...
429,imgflip_1472,surprise;shock;disbelief,awe;fear,none
430,imgflip_1508,dark;serious,disapproval;disappointment,humor;irony;juxtaposition
431,imgflip_1535,happy;relaxed,warm;pleasant;inviting;friendly,none
432,imgflip_1774,enjoyment;happiness;joy;amusement;comfortable;...,positive,exaggerate


In [189]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
def setofcategory(text):
    text = str(text)
    text = text.lower()
    # remove spaces
    text = text.strip()
    # split by ;
    text = text.split(';')
    # get set
    text = set(text)
    txt = ''
    for i in text:
        word = lemmatizer.lemmatize(i)
        if txt == '':
            txt = word
        else:
            txt += ';' + word
    return txt

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [190]:
data['emotion'] = data['emotion'].apply(lambda x: setofcategory(x))
data['sentiment'] = data['sentiment'].apply(lambda x: setofcategory(x))
data['humor'] = data['humor'].apply(lambda x: setofcategory(x))
data.to_csv(dirPath, index=False)

In [191]:
emotion_df = pd.DataFrame()
emotion_df['image_id'] = data['image_id']
# emotion_df['sentiment'] = data['sentiment']
emotion_df

,image_id
0,bokete_94229
1,imgflip_0
2,imgflip_1
3,imgflip_2
4,imgflip_3
...,...
429,imgflip_1472
430,imgflip_1508
431,imgflip_1535
432,imgflip_1774


In [5]:
oxford = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford.csv')
mcdonald = pd.read_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv')
sonic = pd.read_csv('../Data/Instagram/Minigpt4_sonicdrivein.csv')

In [6]:
oxford

,image_id,emotion,sentiment,humor
0,bokete_94229,seriousness;introspection,melancholy;reflection,amusement;irony
1,imgflip_0,joyful,energetic,playful
2,imgflip_1,sad;confused;happy,positive;negative;neutral,none;irony;sarcasm
3,imgflip_2,surprise;frustration;sadness;curiosity,playfulness;anticipation;melancholic,exaggeration;none;irony
4,imgflip_3,intensity;anger;violence;fear,heroism;villainy;gritty;dark,irony
...,...,...,...,...
429,imgflip_1472,surprise;shock;disbelief,awe;fear,none
430,imgflip_1508,dark;serious,disapproval;disappointment,humor;irony;juxtaposition
431,imgflip_1535,happy;relaxed,warm;pleasant;inviting;friendly,none
432,imgflip_1774,enjoyment;happiness;joy;amusement;comfortable;...,positive,exaggerate


In [7]:
emotion_oxford = pd.DataFrame()
emotion_mcdonald = pd.DataFrame()
emotion_sonic = pd.DataFrame()
emotion_oxford['image_id'] = oxford['image_id']
emotion_mcdonald['image_id'] = mcdonald['image_id']
emotion_sonic['image_id'] = sonic['image_id']
all_emotion = set()
for i in range(oxford.shape[0]):
    text = str(oxford['emotion'][i])
    text = text.split(';')
    temp = set(text)
    all_emotion = all_emotion.union(temp)
for i in range(mcdonald.shape[0]):
    text = str(mcdonald['emotion'][i])
    text = text.split(';')
    temp = set(text)
    all_emotion = all_emotion.union(temp)
for i in range(sonic.shape[0]):
    text = str(sonic['emotion'][i])
    text = text.split(';')
    temp = set(text)
    all_emotion = all_emotion.union(temp)
if '' in all_emotion:
    print(len(all_emotion))
    #delete empty emotion
    all_emotion.remove('')
    print('all emotion are empty')
    print(len(all_emotion))

966
all emotion are empty
965


In [6]:
for text in all_emotion:
    if 'energ' in text:
        print(text)

energy
energized
energizing
energy.
energetic


In [221]:
temp = pd.DataFrame()
temp['emotion'] = list(all_emotion)
# temp.to_csv('../Data/Oxford_HIC/all_sentiment.csv', index=False)
temp.to_csv('../Data/Oxford_HIC/all_emotion.csv', index=False)

In [46]:
# 建立 one-hot 編碼欄位
for emotion in all_emotion:
    # if empty  1 or 0  if not emp
    emotion_oxford[emotion] = oxford['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
    emotion_mcdonald[emotion] = mcdonald['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
    emotion_sonic[emotion] = sonic['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
print(emotion_oxford.shape, emotion_mcdonald.shape, emotion_sonic.shape)


C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_oxford[emotion] = oxford['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_mcdonald[emotion] = mcdonald['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:6: PerformanceWarning: DataFrame is highly fr

(434, 968) (787, 968) (1522, 968)


C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_sonic[emotion] = sonic['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_oxford[emotion] = oxford['emotion'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\user\AppData\Local\Temp\ipykernel_7904\620825588.py:5: PerformanceWarning: DataFrame is highly fragment

In [42]:
emotion_oxford[emotion_oxford['energetic']== 1].shape

(2, 968)

In [47]:
def emotion_categorize(df, category):
    for text in all_emotion:
        if re.findall(f'\A{category}\w*', text):
            if df[text] == 1:
                return 1
    return 0
def remove_sub_category(df, category, fullName):
    for text in all_emotion:
        if re.findall(f'\A{category}\w*', text) and fullName != text:
            df.drop([text], axis=1, inplace=True)
    return df

In [278]:
word = 'mys'
for text in all_emotion:
    if re.findall(f'\A{word}\w*', text):
        print(text)

mystery
mysterious


In [251]:
category_list = ['energ','abandon','activ','adven','affection','aggress','alert','amus','annoy','anx','appreciat','associat','bond','bore','bright','calm','careful','casual','celebrat','challeng','chao','cheer','cold','comfor','competiti','complet','concern','concentrat','confident','confus','contemplat','content','cool','coz','crav','creep','curio','cute','dark','deep','delight','depressed','desolat','desp','detach','determin','difficult','disappoint','disconnect','disgust','disinterest','distract','drama','dynam','eager','eeri','embarrass','empath','empower','empt','energ','engag','enjoy','entertain','enthus','excit','exhaust','familiar','fear','festiv','freedom','frustrat','ful','fun','gather','gloomy','happ','helpless','humor','hung','indulgen','intens','interest','intima','intimidati','intrigu','invit','iron','irrita','isolat','joy','lighthearted','lone','long','melancho','mischie','motivat']

fullNameList = ['energetic','abandoned','active','adventure','affection','aggression','alert','amused','annoyed','anxious','appreciative','associated','bond','bored','bright','calm','careful','casual','celebrate','challenge','chaos','cheerful','cold','comfort','competitive','completeness','concern','concentrated','confident','confused','contemplative','content','cool','cozy','craving','creepy','curious','cute','dark','deep','delight','depressed','desolate','despair','detached','determined','difficult','disappointed','disconnected','disgust','disinterest','distracted','dramatic','dynamic','eager','eerie','embarrassed','empathy','empowering','empty','energetic','engaged','enjoy','entertaining','enthusiastic','excited','exhausted','familiar','fear','festive','freedom','frustrated','fulfilled','fun','gathered','gloomy','happy','helpless','humor','hunger','indulgent','intense','interested','intimate','intimidating','intrigue','inviting','ironic','irritation','isolated','joy','lighthearted','lonely','longing','melancholy','mischief','motivated']
print(len(category_list), len(fullNameList))

86 86


In [48]:
emotion_oxford['energetic'] = emotion_oxford.apply(lambda x: emotion_categorize(x, 'energ'), axis=1)
emotion_oxford = remove_sub_category(emotion_oxford, 'energ', 'energetic')


(434, 964)

In [202]:
emotion_oxford.to_csv('../Data/Oxford_HIC/Minigpt4_Oxford_emotion.csv', index=False)
emotion_mcdonald.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland_emotion.csv', index=False)
emotion_sonic.to_csv('../Data/Instagram/Minigpt4_sonicdrivein_emotion.csv', index=False)

In [205]:
emotion = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_emotion.csv')
sentiment = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_sentiment.csv')
humor = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_humor.csv')
print(emotion.shape, sentiment.shape, humor.shape)
emotion['sum'] = emotion.iloc[:, 1:].sum(axis=1)
sentiment['sum'] = sentiment.iloc[:, 1:].sum(axis=1)
humor['sum'] = humor.iloc[:, 1:].sum(axis=1)
print(max(emotion['sum']), max(sentiment['sum']), max(humor['sum']))

(434, 968) (434, 1112) (434, 812)
15 12 14


In [133]:
from string import punctuation

import pandas as pd
from sklearn.model_selection import train_test_split


# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Minigpt4_Oxford.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv'


data = pd.read_csv(dirPath)
# data['chat'] = ''
data['chat'] = data['chat'].str.lower()
# data['chat'] = data.apply(lambda x: x['chat'] if x['done'] != 'O' else '', axis=1)
# if emotion == 'nan':emotion = ''
# data['emotion'] = data['emotion'].apply(lambda x: '' if x == 'nan' else x)
# data['sentiment'] = data['sentiment'].apply(lambda x: '' if x == 'nan' else x)
# data['humor'] = data['humor'].apply(lambda x: '' if x == 'nan' else x)

# data['emotion'] = ''
# data['sentiment'] = ''
# data['humor'] = ''
# replace , with ; in emotion, sentiment, humor
# data['emotion'] = data['emotion'].str.replace(',', ';')
# data['sentiment'] = data['sentiment'].str.replace(',', ';')
# data['humor'] = data['humor'].str.replace(',', ';')
# data.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv', index=False)
# data


In [94]:
data

,image_id,chat,done,emotion,sentiment,humor
0,mcdonalds_switzerland_0,NaN,O,calm;peaceful;serene,happy;content;relaxed,none
1,mcdonalds_switzerland_1,NaN,O,loneliness;isolation;contemplation,solitude;melancholy;serenity,unexpected;contemplation;humorous;face;confusi...
2,mcdonalds_switzerland_2,NaN,O,casual;happy;relaxed,positive;peaceful;serene,ironic;playful;witty
3,mcdonalds_switzerland_3,NaN,O,disgust;sadness,negative,irony;sarcasm
4,mcdonalds_switzerland_4,NaN,O,serenity;joy;calmness;confusion;relaxation,hunger;satisfaction;rush,playfulness;wordplay;irony
...,...,...,...,...,...,...
782,mcdonalds_switzerland_2064,NaN,O,calm;pleasant;refreshment,cheerful;sunny;relaxation;bright,casual;atmosphere;playful
783,mcdonalds_switzerland_2065,NaN,O,hunger;satisfaction;contentment,family;nostalgia;breakfast,playful
784,mcdonalds_switzerland_2069,"i'm not able to see an image, so i cannot prov...",X,joyful;happy;camaraderie;happiness;cheerful;sm...,NaN,dressed;playful
785,mcdonalds_switzerland_2075,NaN,O,curiosity;satisfied;happy;joy;excitement,comfort;foodie,playfulness;irony;unconventional;satire;playful


In [130]:

def setofcategory(text):
    if text == 'nan':
        return ''
    text = str(text)
    text = text.lower()
    # remove spaces
    text = text.strip()
    # split by ;
    text = text.split(';')
    # get set
    text = set(text)
    txt = ''
    for t in text:
        if txt == '':
            txt += t
        else:
            txt += ';'+t
    return txt


In [131]:
data['emotion'] = data['emotion'].apply(setofcategory)
data['sentiment'] = data['sentiment'].apply(setofcategory)
data['humor'] = data['humor'].apply(setofcategory)
data

,image_id,chat,done,emotion,sentiment,humor
0,mcdonalds_switzerland_0,NaN,O,calm;peaceful;serene,happy;content;relaxed,none
1,mcdonalds_switzerland_1,NaN,O,loneliness;isolation;contemplation,solitude;melancholy;serenity,unexpected;contemplation;humorous;face;confusi...
2,mcdonalds_switzerland_2,NaN,O,casual;happy;relaxed,positive;peaceful;serene,ironic;playful;witty
3,mcdonalds_switzerland_3,NaN,O,disgust;sadness,negative,irony;sarcasm
4,mcdonalds_switzerland_4,NaN,O,serenity;joy;calmness;confusion;relaxation,hunger;satisfaction;rush,playfulness;wordplay;irony
...,...,...,...,...,...,...
782,mcdonalds_switzerland_2064,NaN,O,calm;pleasant;refreshment,cheerful;sunny;relaxation;bright,casual;atmosphere;playful
783,mcdonalds_switzerland_2065,NaN,O,hunger;satisfaction;contentment,family;nostalgia;breakfast,playful
784,mcdonalds_switzerland_2069,NaN,O,joyful;happy;camaraderie;happiness;cheerful;sm...,positive;happy;enjoying,dressed;playful
785,mcdonalds_switzerland_2075,NaN,O,curiosity;satisfied;happy;joy;excitement,comfort;foodie,playfulness;irony;unconventional;satire;playful


In [132]:
data.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv')

In [125]:
import re
def addwhat(text, categorylist):
    if ':' in text:
        text = text.split(":")[0]

    elif '-' in text:
        text = text.split("-")[0]
    if '.' in text:
        text = text.split(".")[1]
        text = text.strip()
    # remove punctuation and spaces
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    print(f'text: {text}')
    # check if only one word

    if ' ' in text:
        return categorylist
    if categorylist != '':
        categorylist += ';' + text
    else:
        categorylist = text
    return categorylist

def categorize(text, emotion, sentiment, humor):
    if text == 'nan':
        return emotion, sentiment, humor
    print("==")
    split1 = text.split("\n\n")
    if len(split1) == 1:
        split1 = text.split("\r\n\r\n")
    category = ''
    if emotion == 'nan':
        emotion = ''
    if sentiment == 'nan':
        sentiment = ''
    if humor == 'nan':
        humor = ''
    for i in range(len(split1)):
        # print(category, i)
        print(split1[i])
        if category == '':
            if 'emotion' in split1[i]:
                category = 'emotion'
                continue
            elif 'sentiment' in split1[i]:
                category = 'sentiment'
                continue
            elif 'humor' in split1[i]:
                category = 'humor'
                continue
        else:
            if '*' in split1[i] or '.' in split1[i]:
                split2 = split1[i].split("\r\n")
                if len(split2) == 1:
                    split2 = split1[i].split("\n")
                for j in range(len(split2)):
                    if '*' in split1[i] or '.' in split1[i]:
                        if category == 'emotion':
                            emotion = addwhat(split2[j], emotion)
                        elif category == 'sentiment':
                            sentiment = addwhat(split2[j], sentiment)
                        elif category == 'humor':
                            humor = addwhat(split2[j], humor)
                category = ''
    return emotion, sentiment, humor


In [126]:
data['chat'][11]

nan

In [127]:
categorize(str(data['chat'][11]), str(data['emotion'][11]), str(data['sentiment'][11]), str(data['humor'][11]))

('calmness;wonder;curiosity', 'peaceful', 'unexpected;playful')

In [128]:
data['emotion'], data['sentiment'], data['humor']  = zip(*data.apply(lambda x: categorize(str(x['chat']), str(x['emotion']), str(x['sentiment']), str(x['humor'])), axis=1))

==
the image shows a cat pawprint with the words "meow, even a cat can use order and pay" written above it.
the emotion of the image is playful and lighthearted. it suggests that cats are capable of using order and pay apps just like humans do. the sentiment of the image is positive and humorous, implying that cats are intelligent and capable of using technology.
the key elements of the image are:
* the cat's pawprint, which represents the animal's ability to interact with the world around it.
* the words "meow, even a cat can use order and pay," which convey a sense of humor and playfulness.
* the background of the image, which is orange and provides a lighthearted tone to the overall design.
text: 
text: 
text: 
==
the image depicts a scorpion, a type of insect with a long, segmented body and a tail with a stinger on the end. it is often associated with the sign of scorpio in astrology and is said to be symbolic of strength, passion, and resilience.
here are some elements of the emot

In [129]:
# if emotion, sentiment, humor are all filled done = 'O'
# if emotion, sentiment, humor are all empty done = 'X'
# if emotion, sentiment, humor are not all empty done = 'P'
def checkdone(emotion, sentiment, humor, done):
    if emotion != '' and sentiment != '' and humor != '':
        return 'O'
    else:
        return done

data['done'] = data.apply(lambda x: checkdone(x['emotion'], x['sentiment'], x['humor'], x['done']), axis=1)
data['done'].value_counts()


done
O       783
done      2
X         2
Name: count, dtype: int64

In [56]:
# data.to_csv('../Data/Oxford_HIC/Minigpt4_Oxford.csv', index=False)
data.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv', index=False)

In [59]:
# mix two file
data1 = pd.read_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv')
data2 = pd.read_csv('../Data/Instagram/Minigpt4_2_mcdonalds_switzerland.csv')

In [60]:
data2['emotion1'] = data1['emotion']
data2['sentiment1'] = data1['sentiment']
data2['humor1'] = data1['humor']

In [61]:
def addup(category, category1):
    category = str(category)
    category1 = str(category1)
    # print(category, category1, category == 'nan', category1 == 'nan')
    if category == 'nan' and category1 == 'nan':
        return ''
    elif category == 'nan' or category == category1:
        return category1
    elif category1 == 'nan':
        return category
    return category + ';' + category1
data2['emotion'] = data2.apply(lambda x: addup(x['emotion'], x['emotion1']), axis=1)
data2['sentiment'] = data2.apply(lambda x: addup(x['sentiment'], x['sentiment1']), axis=1)
data2['humor'] = data2.apply(lambda x: addup(x['humor'], x['humor1']), axis=1)
data2

,Unnamed: 0,image_id,chat,done,emotion,sentiment,humor,emotion1,sentiment1,humor1
0,0,mcdonalds_switzerland_0,the image shows a horse standing in front of a...,X,,,,NaN,NaN,NaN
1,1,mcdonalds_switzerland_1,"sure! here are the emotions, sentiment, and hu...",done,loneliness;contemplation;isolation,serenity;solitude;melancholy,,loneliness;contemplation;isolation,serenity;solitude;melancholy,NaN
2,2,mcdonalds_switzerland_2,"the image shows a person sitting on a bench, w...",O,casual;happy;relaxed,positive;serene;peaceful,playful;ironic;witty,NaN,NaN,NaN
3,3,mcdonalds_switzerland_3,the image shows a cartoon face with different ...,O,disgust;sadness,negative,irony;sarcasm,NaN,NaN,NaN
4,4,mcdonalds_switzerland_4,the image is a kitchen scene with an employee ...,X,sentiment,,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
782,782,mcdonalds_switzerland_2064,the image shows a glass of orange juice with a...,X,,,,NaN,NaN,NaN
783,783,mcdonalds_switzerland_2065,the image shows a plate with three pancakes si...,O,hunger;contentment;satisfaction;hunger;satisfa...,breakfast;family;nostalgia,playful,hunger;satisfaction,family;nostalgia,NaN
784,784,mcdonalds_switzerland_2069,the image shows a group of people dressed in r...,X,,,,NaN,NaN,NaN
785,785,mcdonalds_switzerland_2075,the image depicts a young girl holding a white...,done,curiosity;joy;excitement;satisfied;happy,comfort;foodie,playfulness;satire;irony;unconventional;playful,satisfied;happy,comfort;foodie,unconventional;playful


In [64]:
# drop emotion1, sentiment1, humor1
# data2 = data2.drop(columns=['emotion1', 'sentiment1', 'humor1'])
# data2 = data2.drop(columns=['funny_score'])
sumofdata = 0
def checkdone(emotion, sentiment, humor, done):
    global sumofdata
    if emotion != '' and sentiment != '' and humor != '':
        return 'O'
    else:
        sumofdata += 1
        return done

data2['done'] = data2.apply(lambda x: checkdone(x['emotion'], x['sentiment'], x['humor'], x['done']), axis=1)
print(sumofdata)
a = data2['done'].value_counts()
print(a)

385
done
O       402
X       194
done    191
Name: count, dtype: int64


In [65]:
data2.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv', index=False)


In [86]:
from string import punctuation

import pandas as pd
from sklearn.model_selection import train_test_split


# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'


data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()

In [80]:
dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'

datall = pd.read_csv(dirPath)
datall['caption'] = datall['caption'].str.lower()

In [31]:
import json
from collections import defaultdict
# with open('C:/Users/user/fiftyone/coco-2014/raw/captions_val2014.json', 'r') as f:
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_train2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
imageList = list()
captionList = list()
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_val2014.json', 'r') as f:
# with open('C:/Users/user/fiftyone/coco-2014/raw/captions_train2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)

In [32]:
import pandas as pd
data= pd.DataFrame()
data['image_id'] = imageList
data['caption'] = captionList
data

,image_id,caption
0,318556,A very clean and well decorated empty bathroom
1,116100,A panoramic view of a kitchen and all of its a...
2,318556,A blue and white bathroom with butterfly theme...
3,116100,A panoramic photo of a kitchen and dining room
4,379340,A graffiti-ed stop sign across the street from...
...,...,...
616762,401092,A plate of food and a beverage are on a table.
616763,401092,This is an open faced sandwich with several co...
616764,555904,People eating in a restaurant near wine bottles.
616765,6177,The scissors with black handles are sitting open.


In [87]:
#compute the min, max, mean, and variance of the number of tokens in each caption
import nltk
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
import torch
inDictSet = set()
outDictSet = set()
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")

def inDictionary(caption):
    caption = caption.lower()
    split = nltk.word_tokenize(caption)
    # split = caption.split()
    inCount = 0
    outCount = 0
    for word in split:
        if word in inDictSet:
            inCount += 1
        elif word in outDictSet:
            outCount += 1
        else:
            token = tokenizer(word, return_tensors="pt").input_ids
            if token.shape[1] == 1:
                inDictSet.add(word)
                inCount += 1
            else:
                outDictSet.add(word)
                outCount += 1
    print(f'inCount: {inCount}, outCount: {outCount}')
    return inCount, outCount


In [88]:
data['inDictCount'], data['outDictCount'] = zip(*data['caption'].map(inDictionary))

inCount: 11, outCount: 2
inCount: 7, outCount: 1
inCount: 14, outCount: 4
inCount: 9, outCount: 0
inCount: 7, outCount: 0
inCount: 5, outCount: 0
inCount: 6, outCount: 0
inCount: 5, outCount: 1
inCount: 13, outCount: 5
inCount: 5, outCount: 0
inCount: 5, outCount: 0
inCount: 9, outCount: 1
inCount: 14, outCount: 1
inCount: 3, outCount: 1
inCount: 7, outCount: 0
inCount: 10, outCount: 0
inCount: 5, outCount: 7
inCount: 2, outCount: 1
inCount: 13, outCount: 0
inCount: 13, outCount: 2
inCount: 5, outCount: 1
inCount: 12, outCount: 0
inCount: 19, outCount: 1
inCount: 10, outCount: 2
inCount: 6, outCount: 2
inCount: 2, outCount: 1
inCount: 5, outCount: 2
inCount: 5, outCount: 2
inCount: 5, outCount: 2
inCount: 14, outCount: 2
inCount: 5, outCount: 0
inCount: 5, outCount: 6
inCount: 4, outCount: 0
inCount: 5, outCount: 0
inCount: 19, outCount: 2
inCount: 6, outCount: 1
inCount: 1, outCount: 2
inCount: 8, outCount: 1
inCount: 5, outCount: 0
inCount: 10, outCount: 1
inCount: 8, outCount: 3
inC

In [89]:
print(data.shape)
data.describe()


(2208, 6)


,funny_score,inDictCount,outDictCount
count,2208.000000,2208.000000,2208.000000
mean,0.223053,16.007246,3.511322
std,0.409945,14.642854,3.672213
min,0.000000,0.000000,0.000000
25%,0.000000,8.000000,1.000000
50%,0.000000,13.000000,3.000000
75%,0.000000,20.000000,4.000000
max,1.000000,358.000000,62.000000


In [91]:
#keep data with outDictCount = 0
new_data = data[data['outDictCount'] <= 2]
print(new_data.shape)
new_data = data[data['inDictCount'] <= 60]
print(new_data.shape)
new_data.describe()

(1072, 6)
(2172, 6)


,funny_score,inDictCount,outDictCount
count,2172.000000,2172.000000,2172.000000
mean,0.226750,14.905617,3.274862
std,0.412314,10.179360,2.986297
min,0.000000,0.000000,0.000000
25%,0.000000,8.000000,1.000000
50%,0.000000,13.000000,3.000000
75%,0.000000,20.000000,4.000000
max,1.000000,60.000000,25.000000


In [69]:
image_id_counts = new_data['image_id'].value_counts()
######################################################################################################
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    if i == 9:
        for j in range(10):
            if j == 9:
                x = image_id_counts[image_id_counts >= 5]
                x = x[x < 10]
                sum += len(x)
                print(f' {5:4d} <= caption < {10:4d} --- {len(x):6d} --- {sum:6d}')
                x = image_id_counts[image_id_counts >= 0]
                x = x[x < 5]
                sum += len(x)
                print(f' {0:4d} <= caption < {5:4d} --- {len(x):6d} --- {sum:6d}')
            else:
                x = image_id_counts[image_id_counts >= (10-1-j)*10]
                x = x[x < (10-j)*10]
                sum += len(x)
                print(f' {(10-1-j)*10:4d} <= caption < {(10-j)*10:4d} --- {len(x):6d} --- {sum:6d}')
    else:
        x = image_id_counts[image_id_counts >= (10-1-i)*100]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')

           Total image counts: 115932
 1000 <= caption        ---    169 ---    169
  900 <= caption < 1000 ---      6 ---    175
  800 <= caption <  900 ---      9 ---    184
  700 <= caption <  800 ---     13 ---    197
  600 <= caption <  700 ---     18 ---    215
  500 <= caption <  600 ---     28 ---    243
  400 <= caption <  500 ---     39 ---    282
  300 <= caption <  400 ---     55 ---    337
  200 <= caption <  300 ---    111 ---    448
  100 <= caption <  200 ---   1074 ---   1522
   90 <= caption <  100 ---    119 ---   1641
   80 <= caption <   90 ---    155 ---   1796
   70 <= caption <   80 ---    193 ---   1989
   60 <= caption <   70 ---    272 ---   2261
   50 <= caption <   60 ---    508 ---   2769
   40 <= caption <   50 ---   1000 ---   3769
   30 <= caption <   40 ---   2541 ---   6310
   20 <= caption <   30 ---   8685 ---  14995
   10 <= caption <   20 ---  36691 ---  51686
    5 <= caption <   10 ---  40467 ---  92153
    0 <= caption <    5 ---  23779 --- 115

In [78]:
print("shape of data: ", new_data.shape)
print()
image_id_counts = new_data['image_id'].value_counts()
print(len(image_id_counts))
######################################################################################################
valid_image_ids = image_id_counts[image_id_counts >= 1].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
train = (
    filtered_data.sort_values(by=['image_id','funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1)
)
print("shape of train: ", train.shape)
# train.to_csv('../Data/Oxford_HIC/Only10_split_simCOCO_oxford_hic_data.csv', index=False)
train.describe()


shape of data:  (523, 6)

523
shape of valid_image_ids:  (523,)
shape of filtered_data:  (523, 6)
shape of train:  (523, 6)


,funny_score,inDictCount,outDictCount
count,523.000000,523.000000,523.000000
mean,0.414914,11.430210,1.229446
std,0.481385,6.483135,0.748519
min,0.000000,0.000000,0.000000
25%,0.000000,7.000000,1.000000
50%,0.000000,10.000000,1.000000
75%,1.000000,15.000000,2.000000
max,1.000000,33.000000,2.000000


In [103]:
train[train['caption'] == 'nan']
print("shape of train: ", train.shape)
train=train[train['caption'] != 'nan']
print("shape of train: ", train.shape)

shape of train:  (460765, 6)
shape of train:  (460764, 6)


In [104]:
train.to_csv('../Data/Oxford_HIC/Only5_simCOCO_oxford_hic_data.csv', index=False)

In [ ]:
print("shape of data: ", data.shape)
print()
image_id_counts = data['image_id'].value_counts()
######################################################################################################
valid_image_ids = image_id_counts[image_id_counts >= 7].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", data.shape)
train = (
    filtered_data.sort_values(by=['image_id'], ascending=[True, False])
    .groupby('image_id')
    .head(10)
)
print("shape of train: ", train.shape)
train['inDictCount'], train['outDictCount'] = zip(*train['caption'].map(inDictionary))
print(train.shape)
train.describe()

In [ ]:
#keep data with outDictCount = 0
data = data[data['outDictCount'] == 0]
print(data.shape)
data.to_csv('../Data/Oxford_HIC/NoOutDict_oxford_hic_data.csv', index=False)

In [ ]:
data=pd.read_csv('../Data/Oxford_HIC/NoOutDict_oxford_hic_data.csv')

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(10)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/NoOutDictTop10_oxford_hic_data.csv', index=False)

# Generate data

In [30]:
from string import punctuation
import pandas as pd
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'

data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()

In [32]:
import pandas as pd
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
import albumentations
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

class NLPTransform(BasicTransform):
    """ Transform for nlp task."""

    @property
    def targets(self):
        return {"data": self.apply}

    def update_params(self, params, **kwargs):
        if hasattr(self, "interpolation"):
            params["interpolation"] = self.interpolation
        if hasattr(self, "fill_value"):
            params["fill_value"] = self.fill_value
        return params

    def get_sentences(self, text, lang='en'):
        return sent_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [33]:
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
import albumentations
import torch
import re
import string

class LMmask(NLPTransform):

    def __init__(self, mask_num = 1, tokenizer_name='bert-base-uncased', model_name='bert-base-uncased'):
        self.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’'
        self.mask_num = mask_num
        if tokenizer_name == 'BertTokenizer':
            #   BertTokenizer + BertForMaskedLM  ==> 'bert-base-uncased'
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
            self.model = BertForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 103
        elif tokenizer_name == 'RobertaTokenizer':
            #   RobertaTokenizer + RobertaForMaskedLM  ==> 'FacebookAI/roberta-base'
            self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
            self.model = RobertaForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 50264
        elif tokenizer_name == 'AutoTokenizer':
            # AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)


            self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def apply(self, data, top_k=10):
        new_text = []
        split = nltk.word_tokenize(data)
        for idx, n in enumerate(split):
            words = ''
            not_enough = False
            for i in range(self.mask_num):
                if idx + i >= len(split):
                    not_enough = True
                    break
                if split[idx + i] in self.punctuation:
                    words = words + split[idx + i]
                else:
                    words = words + ' ' + split[idx + i]
            if not_enough:
                continue
            words = re.sub(r'\s+', ' ', words)
            words = words.strip()
            token = self.tokenizer.encode(words, return_tensors="pt")
            single = token.shape[1] == self.mask_num + 2
            if single:
                text = ''
                words = nltk.word_tokenize(words)
                for idx_w, word_split in enumerate(split):
                    if word_split in words and idx_w in range(idx, idx + self.mask_num):
                        words.remove(word_split)
                        text = text + ' ' + self.mask_token
                    elif word_split in self.punctuation:
                        text = text + word_split
                    else:
                        text = text + ' ' + word_split
                text = re.sub(r'\s+', ' ', text)
                text = text.strip()
                # DEFINE SENTENCE
                indices = self.tokenizer.encode(text, add_special_tokens=True, return_tensors='pt')
                # PREDICT MISSING WORDS
                pred = self.model(indices)
                masked_indices = torch.where(indices == self.mask_token_id)[1]
                # TOP 10 PREDICTIONS
                top10 = torch.topk(pred[0][0][masked_indices, :], top_k, axis=1)
                # FILL IN MISSING WORDS
                for i in range(top_k):
                    temp = text
                    for j in range(self.mask_num):
                        if j < top10.indices.shape[0] and i < top10.indices.shape[1]:
                            temp = temp.replace(self.mask_token, self.tokenizer.decode(top10.indices[j][i]), 1)
                    temp = re.sub(r'\s+', ' ', temp)
                    new_text.append(temp)
        return new_text

In [34]:
data.head()

,caption,image_id,funny_score,caption_id
0,consider this a wake up call for more fun. 🥤#l...,sonicdrivein_0,0.0,caption1
1,thx eric for showing me how to shadow,sonicdrivein_1,0.0,caption2
2,@parishilton and @nicolerichie returned to the...,sonicdrivein_2,0.0,caption3
3,its ✨ sonic math ✨ dont question our methods,sonicdrivein_3,0.0,caption4
4,getting ready for my tot girl walk,sonicdrivein_4,0.0,caption5


In [35]:
# complete following code
new_data = pd.DataFrame(columns=['caption', 'image_id', 'funny_score'])

with tqdm(total=data.shape[0]) as progress_bar:
    for i in range(data.shape[0]):
        text = data.caption[i]
        image_id = data.image_id[i]
        funnyscore = data.funny_score[i]
        new_data = pd.concat([new_data, pd.DataFrame([[text, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
        generate_wordcount = len(nltk.word_tokenize(text)) - 3
        # print(f'image_id{image_id}, funnyscore{funnyscore}, generate_wordcount{generate_wordcount}, text{text}')
        if generate_wordcount > 0:
            for j in range(min(generate_wordcount, 5)):
                # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
                lm = LMmask(mask_num = j+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
                temp_sentences = lm.apply(text, top_k=5)

                for sentence in temp_sentences:
                    new_data = pd.concat([new_data, pd.DataFrame([[sentence, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences), "size": new_data.shape})
        progress_bar.update()
new_data['caption'] = new_data['caption'].str.lower()
# remove duplicate data
new_data = new_data.drop_duplicates()

  0%|          | 0/2208 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_11496\2493366767.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([[text, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
100%|██████████| 2208/2208 [4:06:50<00:00,  6.71s/it, j=4, k=15, size=(519183, 3)]   


In [36]:
new_data

,caption,image_id,funny_score
0,consider this a wake up call for more fun. 🥤#l...,sonicdrivein_0,0.0
1,consider this a wake up call for more fun. 🥤# ...,sonicdrivein_0,0.0
2,is this a wake up call for more fun. 🥤# livefr...,sonicdrivein_0,0.0
3,call this a wake up call for more fun. 🥤# live...,sonicdrivein_0,0.0
4,make this a wake up call for more fun. 🥤# live...,sonicdrivein_0,0.0
...,...,...,...
519178,sonic ceo cliff hudson interviewing with@ news...,sonicdrivein_2239,0.0
519179,sonic ceo cliff hudson interviewing with@ news...,sonicdrivein_2239,0.0
519180,sonic ceo cliff hudson interviewing with@ news...,sonicdrivein_2239,0.0
519181,sonic ceo cliff hudson interviewing with@ news...,sonicdrivein_2239,0.0


In [37]:
# 計算每個 image_id 的資料數量
image_id_counts = new_data['image_id'].value_counts()
print(f'Number of unique image_id: {len(image_id_counts)}')
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 300].index
print(f'Number of image_id with 300 captions: {len(valid_image_ids)}')
# 篩選原始資料
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print(f'Number of data: {filtered_data.shape[0]}')
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(300)
)
print(f'Number of data: {top_captions.shape[0]}')

Number of unique image_id: 2208
Number of image_id with 300 captions: 447
Number of data: 222368
Number of data: 134100


In [38]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds images: 9/306 , data = 2700/3654/29804
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
# mcdonaldscanada images: 150/843 , data = 45000/86156/179124
# sonicdrivein images: 447/2208 , data = 134100/222368/460574
# wendys images: 14/367 , data = 4200/5846/35727
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_sonicdrivein.csv', index=False)

(460574, 3)


In [76]:
import json
from collections import defaultdict
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_val2014.json', 'r') as f:
# with open('C:/Users/user/fiftyone/coco-2014/raw/captions_train2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
grouped_data = defaultdict(list)
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    grouped_data[image_id].append(caption)
image_id_counts2 = {image_id: len(captions) for image_id, captions in grouped_data.items()}

In [72]:
#check if the number of captions for each image_id is the same
print(len(image_id_counts))
print(len(set(image_id_counts.values())))


82783
3


In [79]:
print(f'                       COCO')
print(f'           Total image counts: {len(image_id_counts)}')
x = [count for count in image_id_counts.values() if count >= 10]
y = [count for count in image_id_counts2.values() if count >= 10]
sum = len(x)+len(y)
print(f'   {10} <= caption        --- {len(x)+len(y):6d} --- {sum:6d}')
for i in range(10):
    x = [count for count in image_id_counts.values() if (10-1-i)*1 <= count < (10-i)*1]
    y = [count for count in image_id_counts2.values() if (10-1-i)*1 <= count < (10-i)*1]
    sum += len(x)+len(y)
    print(f' {(10-1-i)*1:4d} <= caption < {(10-i)*1:4d} --- {len(x)+len(y):6d} --- {sum:6d}')

                       COCO
           Total image counts: 82783
   10 <= caption        ---      0 ---      0
    9 <= caption <   10 ---      0 ---      0
    8 <= caption <    9 ---      0 ---      0
    7 <= caption <    8 ---      4 ---      4
    6 <= caption <    7 ---    324 ---    328
    5 <= caption <    6 --- 122959 --- 123287
    4 <= caption <    5 ---      0 --- 123287
    3 <= caption <    4 ---      0 --- 123287
    2 <= caption <    3 ---      0 --- 123287
    1 <= caption <    2 ---      0 --- 123287
    0 <= caption <    1 ---      0 --- 123287


In [60]:
import pandas as pd
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
name = 'sonicdrivein'
fileName = 'Generate_' + name
dirPath = '../Data/Instagram/' + fileName + '.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
print()
image_id_counts = data['image_id'].value_counts()
######################################################################################################
print(f'             {fileName}')
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 300]
# sum = len(x)
# print(f' {300} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 200]
# x = x[x < 300]
# sum += len(x)
# print(f'{200:4d} <= caption < {300:4d} --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 100]
# x = x[x < 200]
# sum += len(x)
# print(f'{100:4d} <= caption < {200:4d} --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    x = image_id_counts[image_id_counts >= (10-1-i)*100]
    x = x[x < (10-i)*100]
    sum += len(x)
    print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


shape of data:  (460574, 3)

             Generate_sonicdrivein
           Total image counts: 2208
 1000 <= caption        ---     18 ---     18
  900 <= caption < 1000 ---     10 ---     28
  800 <= caption <  900 ---     17 ---     45
  700 <= caption <  800 ---     21 ---     66
  600 <= caption <  700 ---     31 ---     97
  500 <= caption <  600 ---     30 ---    127
  400 <= caption <  500 ---     97 ---    224
  300 <= caption <  400 ---    223 ---    447
  200 <= caption <  300 ---    453 ---    900
  100 <= caption <  200 ---    622 ---   1522
    0 <= caption <  100 ---    686 ---   2208


In [ ]:
text = "You can’t achieve strawberry lemonade until you first get strawberry lemonade followed by strawberry lemonade."
new_data = pd.DataFrame()
for i in range(3):
    # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
    lm = LMmask(mask_num = i+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
# lm = LMmask(mask_num = 1, tokenizer_name='AutoTokenizer', model_name='')
    # 'DataFrame' object has no attribute 'append'

    new_data = pd.concat([new_data, pd.DataFrame(lm.apply(text, top_k=5))], ignore_index=True)
    new_data = pd.concat([new_data, pd.DataFrame(["============================================================="])], ignore_index=True)
new_data
# lm = LMmask(mask_num = 1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
# lm.apply(text, top_k=5)


In [ ]:
new_data.to_csv('./roberta2.csv', index=False)

In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
tokenizer.decode(2226)


In [ ]:
import string
from torch.utils.data import Dataset, DataLoader
import clip
import os
import pandas as pd
import pickle
from torch import nn
import numpy as np
import torch
import torch.nn.functional as nnf
import sys
from typing import Tuple, List, Union, Optional
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    AdamW,
    get_linear_schedule_with_warmup,
)
from transformers import AutoConfig, AutoTokenizer, Gemma2ForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
import PIL.Image
from tqdm import tqdm
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
inwordList = set()
outwordList = set()
progress_counter = tqdm(total=len(data), desc='Counting tokens', position=0, leave=True)
progress_inDict = tqdm(total=len(data), desc='Counting in dict', position=0, leave=True)
def token_counter(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    progress_counter.update(1)
    return len(temp)

def word_in_dict(caption):
    # caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    notTested = temp.difference(inwordList).difference(outwordList)
    notInDictCounter = len(temp.intersection(outwordList))
    # print(f'notInDictCounter: {notInDictCounter}, notTested: {notTested}')
    for word in notTested:
        token = tokenizer(word, return_tensors="pt").input_ids
        # print(f'word: {word}, token: {token}')
        if token.shape[1] == 1:
            inwordList.add(word)
        else:
            notInDictCounter += 1
            outwordList.add(word)
    # progress_inDict.update(1)
    return notInDictCounter

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# 計算每個 caption 的字數
data['token_count'] = data['caption'].apply(token_counter)
progress_counter.close()
# 計算每個 caption 中不在字典內的字數
data['out_of_dict_count'] = data['caption'].apply(word_in_dict)
progress_inDict.close()
print(f'Number of unique words in the dictionary: {len(inwordList)}')
print(f'Number of unique words out of the dictionary: {len(outwordList)}')

In [ ]:
test_tokens = inwordList.union(outwordList)

In [ ]:
train_tokens = inwordList.union(outwordList)

In [ ]:
a = set(test_tokens)
b = set(train_tokens)
c = set()
d = set()
for testDatasetToken in a:
    c.add(testDatasetToken.item())
print(len(c))
for testDatasetToken in b:
    d.add(testDatasetToken.item())
print(len(d))
print(len(c.symmetric_difference(d)))
print(len(c.intersection(d)))

In [ ]:
print(data.shape)
print(data[data['out_of_dict_count'] < 4].shape)
print(data[data['out_of_dict_count'] < 3].shape)
print(data[data['out_of_dict_count'] < 2].shape)
print(data[data['out_of_dict_count'] < 1].shape)
print(data[data['out_of_dict_count'] < 0].shape)
data.describe()

In [ ]:
data = data[data['out_of_dict_count'] < 1]

In [ ]:
# 計算每個 image_id 的資料數量
image_id_counts = data['image_id'].value_counts()
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 1000].index
# 篩選原始資料
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print(len(valid_image_ids))
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1000)
)
print(top_captions.shape)
print(filtered_data.shape)

In [ ]:
print(f'          Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
for i in range(10):

    if i == 9:
        x = image_id_counts[image_id_counts >= 50]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {50:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts >= 10]
        x = x[x < 50]
        sum += len(x)
        print(f' {10:4d} <= caption < {50:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts < 10]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {10:4d} --- {len(x):6d} --- {sum:6d}')
    else:
        x = image_id_counts[image_id_counts >= (10-1-i)*100]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


In [ ]:
# get data differ from data_all and data1500
print(data.shape)
print(filtered_data.shape)
exceptdata = data[~data['image_id'].isin(filtered_data['image_id'])]
print(len(exceptdata['image_id'].value_counts()))
print(exceptdata.shape)
exceptdata.to_csv('../Data/Oxford_HIC/except1000up_oxford_hic_data.csv', index=False)

In [ ]:
filtered_data.to_csv('../Data/Oxford_HIC/1000up_oxford_hic_data.csv', index=False)

In [ ]:
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
whole = pd.read_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv')
unique_image_ids = whole['image_id'].unique()

train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
train = whole[whole['image_id'].isin(train_ids)]
test = whole[whole['image_id'].isin(test_ids)]
print(train.shape, test.shape)

In [ ]:
train_mess = pd.DataFrame()
test_mess = pd.DataFrame()
for image_id, group in whole.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train_mess = pd.concat([train_mess, train_split])
    test_mess = pd.concat([test_mess, test_split])
print(f'train: {train_mess.shape}')
print(f'test: {test_mess.shape}')

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504']
train_data = train[train['image_id'].isin(train_image)]
test_image = ['bokete_100174', 'imgflip_834']
test_data = test[test['image_id'].isin(test_image)]
print(train_data.shape, test_data.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_oxford_hic_data.csv', index=False)

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
train_data_mess = train_mess[train_mess['image_id'].isin(train_image)]
test_image =  ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
test_data_mess = test_mess[test_mess['image_id'].isin(test_image)]
print(train_data_mess.shape, test_data_mess.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_mess_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_mess_oxford_hic_data.csv', index=False)

In [ ]:
train_data_mess

In [ ]:
# list same caption in train_data and test_data_mess
a = test_data[test_data['image_id'] == 'imgflip_130']
b = test_data_mess[test_data_mess['image_id'] == 'imgflip_130']
x = set(a['caption']).intersection(set(b['caption']))
for i in x:
    if '' in i:
        print(i)
# 2spbgym,climb a mountain? pff, i have wings...
# all-the-things,go to a pizza buffet eat all the pizza
# imgflip_0,12 dollars; 11 dollars with 1 dollar shipping
# imgflip_1033,I POUR MILK BEFORE CEREAL
# imgflip_11,ME WAITING FOR MY INTERNET TO RECONNECT
# imgflip_117,calling the teacher mom
# imgflip_16,IF SOMEONE DIES IN THE LIVING ROOM... IS IT STILL CALLED THE LIVING ROOM?
# imgflip_189,you; losing a few seconds of your life looking at this
# imgflip_23,me: gets up and starts clapping because the chiefs won; the guy who has been pushing my wheelchair for 10 years
# imgflip_504,THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS

# i-love-coloring-kid,she started writing notes !!
# imgflip_130,WHEN YOUR FRIEND; DOSENT LIKE ROOT BEER

In [ ]:
a

In [ ]:
img_names = data['image_id'].unique()
len(img_names)

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(50)
)
top_captions.shape

In [ ]:
train_data = pd.DataFrame()
test_data = pd.DataFrame()
only = pd.DataFrame()
for image_id, group in top_captions.groupby("image_id"):
    if group.shape[0] < 50:
        continue
    only = pd.concat([only, group])
    # train, test = train_test_split(group, test_size=0.2, random_state=42)
    # train_data = pd.concat([train_data, train])
    # test_data = pd.concat([test_data, test])
print(only.shape)

In [ ]:
only.to_csv('../Data/Oxford_HIC/Only50_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Top10_oxford_hic_data.csv', index=False)

In [ ]:
# 獲取唯一的 image_id
unique_image_ids = top_captions['image_id'].unique()
print(unique_image_ids.shape)
# 將 image_id 拆分為 80% 訓練集和 20% 測試集
train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)

# 根據拆分的 image_id 選取資料
train_data = top_captions[top_captions['image_id'].isin(train_ids)]
test_data = top_captions[top_captions['image_id'].isin(test_ids)]
print(train_data.shape, test_data.shape)

In [ ]:
unique_image_ids.shape[0]/3

In [ ]:
a = unique_image_ids[:30000]
print(a.shape)

In [ ]:
dirPath = '../Data/Oxford_HIC/Only1200_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
######################################################################################################
train = pd.DataFrame()
test = pd.DataFrame()
for image_id, group in data.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train = pd.concat([train, train_split])
    test = pd.concat([test, test_split])
print(f'train: {train.shape}')
print(f'test: {test.shape}')
######################################################################################################
# unique_image_ids = data['image_id'].unique()
# # unique_image_ids = unique_image_ids[:30000]
# # unique_image_ids, rest = train_test_split(unique_image_ids, test_size=0.7, random_state=42)
# # print(unique_image_ids.shape)
# train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
# train = data[data['image_id'].isin(train_ids)]
# test = data[data['image_id'].isin(test_ids)]
# print(train.shape, test.shape)

In [ ]:
train=train.reset_index()
test=test.reset_index()

In [ ]:



train_image = ['imgflip_0', 'imgflip_101', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_47', 'imgflip_504']
train_text = ['12 dollars; 11 dollars with 1 dollar shipping'
              ,'I HAD A GIRLFRIEND; AAAAAAND ITS GONE'
              ,'I POUR THE CEREAL AFTER I POUR THE MILK'
              ,'WAITING FOR MY PHONE TO GET  TO 100%'
              ,'You when you have over one test at school in a day'
              ,'IF SOMEONE WANTS TO KILL YOU; GO TO A LIVING ROOM'
              ,'you; eating 5 pounds of cheese; every day; your stomach'
              ,'Me:stands up to stretch my legs; The person who had been pushing my wheelchair for the last 26 years'
              ,'Me: Opens door for some fresh air; Everyone else in the submarine:'
              ,'THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS']
# imgflip_130,WHEN YOU SEE PICS OF YOUR FRIENDS HANGING OUT; BUT YOU WEREN'T INVITED
# imgflip_659,When the teacher uses your voice recording on the homework as an example
test_image = ['imgflip_130', 'imgflip_659']
test_text = ['0 VIEWS 5 DISLIKES'
              ,'when the mobile game ad is so laggy that it crashes your game and you lose out on a reward:']

tokens_list = []
mask_list = []
prefix_list = []
train_gt = []
train_caption = dict()
train_image_id_list = []
# mess = set()
mess_a = set()
mess_b =set()
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
print(train.shape)
for i in range(train.shape[0]):
    caption = train['caption'][i]
    image_id = train['image_id'][i]
    # if image_id == 'imgflip_117':#and 'GONE' in caption:
    #     notmess.add(caption)
    #     print(f"Image ID: {image_id}, Caption: {caption}")
# train_image = ['', 'imgflip_101', '','', 'imgflip_117', '', '','', '', '']


    if image_id in train_image and caption in train_text:
        print(f"Image ID: {image_id}, Caption: {caption}")
print("===================================================================================")
for i in range(test.shape[0]):
    caption = test['caption'][i]
    image_id = test['image_id'][i]
    if image_id == 'imgflip_130':#and 'GONE' in caption:
        mess_a.add(caption)
    if image_id == 'imgflip_659':
        mess_b.add(caption)
    if image_id in test_image and caption in test_text:
        print(f"Image ID: {image_id}, Caption: {caption}")


In [ ]:
set.intersection(notmess_b, mess_b)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-2.7B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-2.7B")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")